In [37]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

from tqdm import tqdm
import time
warnings.filterwarnings('ignore')

# ----------------------------
# Logging
# ----------------------------
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler("debug.log", encoding="utf-8"),  # 파일: 무제한
        logging.StreamHandler()                              # 노트북 셀: 요약만
    ]
)
# ----------------------------

class ClusterBasedForecastingModel:
    """클러스터 기반 예측 모델 - 실제 활용 버전"""
    
    def __init__(self):
        self.menu_clusters = {}
        self.cluster_models = {}
        self.cluster_features = {}
        self.store_patterns = {}

        self.menu_launch_dates = {}  # 메뉴별 출시일
        self.corrected_zero_ratio = None  # 수정된 zero ratio

    def _detect_menu_launches(self, train_df):
        """메뉴별 출시일 감지"""
        
        print("메뉴 출시일 분석 중...")
        
        launch_dates = {}
        pre_launch_count = 0
        
        for (store, menu), group in train_df.groupby(['store', 'menu']):
            group_sorted = group.sort_values('date').reset_index(drop=True)
            
            # 100일+ 연속 0 패턴 찾기
            consecutive_zeros = 0
            for i, row in group_sorted.iterrows():
                if row['sales'] == 0:
                    consecutive_zeros += 1
                else:
                    break
            
            # 100일+ 연속 0이면 출시 전으로 판단
            if consecutive_zeros >= 100:
                first_sale_idx = group_sorted[group_sorted['sales'] > 0].index
                if len(first_sale_idx) > 0:
                    launch_date = group_sorted.loc[first_sale_idx[0], 'date']
                    launch_dates[(store, menu)] = launch_date
                    pre_launch_count += consecutive_zeros
                    print(f"{store}_{menu}: {consecutive_zeros}일 출시 전 → {launch_date}")
        
        self.menu_launch_dates = launch_dates
        
        print(f"총 {len(launch_dates)}개 메뉴의 출시 전 패턴 감지")
        print(f"출시 전 데이터: {pre_launch_count:,}개")
        
        return launch_dates
    
    def _add_pre_launch_flag(self, df):
        """출시 전 플래그 추가"""
        
        df = df.copy()
        df['is_pre_launch'] = 0
        
        for (store, menu), launch_date in self.menu_launch_dates.items():
            mask = (df['store'] == store) & (df['menu'] == menu) & (df['date'] < launch_date)
            df.loc[mask, 'is_pre_launch'] = 1
        
        pre_launch_count = df['is_pre_launch'].sum()
        print(f"출시 전 플래그 적용: {pre_launch_count:,}개 ({pre_launch_count/len(df)*100:.1f}%)")
        
        return df
    
    def _calculate_corrected_zero_ratio(self, train_df):
        """출시 전 제외한 실제 zero ratio 계산"""
        
        # 출시 전이 아닌 데이터만 필터링
        post_launch_df = train_df[train_df['is_pre_launch'] == 0]
        
        zero_count = (post_launch_df['sales'] == 0).sum()
        total_count = len(post_launch_df)
        
        corrected_ratio = zero_count / total_count
        
        print(f"수정된 통계:")
        print(f"  출시 후 데이터: {total_count:,}개")
        print(f"  0 매출: {zero_count:,}개")
        print(f"  실제 zero ratio: {corrected_ratio:.3f} (기존 threshold 0.3 vs 수정 {corrected_ratio:.3f})")
        
        self.corrected_zero_ratio = corrected_ratio
        return corrected_ratio
    
    def classify_menu_stability(self, train_df):
        """메뉴별 안정성 분류"""
        
        menu_stability = {}
        
        for (store, menu), group in train_df.groupby(['store', 'menu']):
            if len(group) < 30:  # 데이터 부족시 제외
                continue
                
            zero_ratio = (group['sales'] == 0).mean()
            total_sales = group['sales'].sum()
            
            # 안정성 분류 기준
            if zero_ratio < 0.1:
                stability = 'very_stable'
            elif zero_ratio < 0.3:
                stability = 'stable'  
            elif zero_ratio < 0.6:
                stability = 'moderate'
            else:
                stability = 'unstable'
                
            menu_stability[(store, menu)] = {
                'stability': stability,
                'zero_ratio': zero_ratio,
                'total_sales': total_sales,
                'avg_sales': group['sales'].mean(),
                'volatility': group['sales'].std() / (group['sales'].mean() + 1e-8)
            }
        
        self.menu_stability_map = menu_stability
        return menu_stability
    
    def analyze_and_cluster_menus(self, train_df):
        """메뉴 클러스터링 및 클러스터별 특성 분석"""
        
        menu_features = []
        menu_names = []
        
        logging.debug("메뉴 클러스터링 시작...")
        
        # 1. 각 메뉴별 특성 추출
        grouped = train_df.groupby(['store', 'menu'])
        for (store, menu), group in tqdm(grouped, desc="메뉴 특성 추출"):
            if len(group) < 20:  # 충분한 데이터가 있는 메뉴만
                continue
                
            # 메뉴별 특성 벡터 생성
            features = {
                'avg_sales': group['sales'].mean(),
                'std_sales': group['sales'].std(),
                'zero_ratio': (group['sales'] == 0).mean(),
                'max_sales': group['sales'].max(),
                'cv': group['sales'].std() / (group['sales'].mean() + 1e-8),  # 변동계수
                'weekend_boost': group[group['is_weekend']]['sales'].mean() / (group['sales'].mean() + 1e-8),
                'seasonality_strength': self._calculate_seasonality(group),
                'trend_strength': self._calculate_trend(group['sales'].values),
            }
            
            # 메뉴명 기반 특성
            menu_lower = str(menu).lower()
            features.update({
                'is_main_dish': int(any(x in menu_lower for x in ['불고기', '갈비', '찌개', '국밥', '정식'])),
                'is_drink': int(any(x in menu_lower for x in ['콜라', '맥주', '소주', '커피', '음료', '차'])),
                'is_premium': int(any(x in menu_lower for x in ['한우', 'aus', '프리미엄'])),
                'is_group': int('단체' in menu_lower),
                'is_brunch': int('브런치' in menu_lower),
            })
            
            menu_features.append(list(features.values()))
            menu_names.append((store, menu))
        
        if len(menu_features) < 5:
            logging.debug("클러스터링에 충분한 메뉴가 없음")
            return {}
        
        # 2. K-means 클러스터링
        from sklearn.cluster import KMeans
        from sklearn.preprocessing import StandardScaler
        
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(menu_features)
        
        # 최적 클러스터 수 결정
        n_clusters = min(6, max(3, len(menu_features) // 8))
        logging.debug(f"클러스터 수: {n_clusters}")
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(features_scaled)
        
        # 3. 클러스터별 메뉴 그룹화 및 특성 분석
        cluster_groups = {}
        cluster_characteristics = {}
        
        for cluster_id in tqdm(range(n_clusters), desc="클러스터 특성 분석"):
            cluster_mask = cluster_labels == cluster_id
            cluster_menus = [menu_names[i] for i in range(len(menu_names)) if cluster_mask[i]]
            cluster_groups[cluster_id] = cluster_menus
            
            # 클러스터 특성 계산
            cluster_data_list = []
            for store, menu in cluster_menus:
                menu_data = train_df[(train_df['store'] == store) & (train_df['menu'] == menu)]
                cluster_data_list.append(menu_data)
            
            if cluster_data_list:
                cluster_combined = pd.concat(cluster_data_list, ignore_index=True)
                
                cluster_characteristics[cluster_id] = {
                    'avg_sales': cluster_combined['sales'].mean(),
                    'zero_ratio': (cluster_combined['sales'] == 0).mean(),
                    'volatility': cluster_combined['sales'].std() / (cluster_combined['sales'].mean() + 1e-8),
                    'weekend_effect': cluster_combined[cluster_combined['is_weekend']]['sales'].mean() / 
                                    (cluster_combined[~cluster_combined['is_weekend']]['sales'].mean() + 1e-8),
                    'menu_count': len(cluster_menus),
                    'dominant_type': self._get_dominant_menu_type(cluster_menus)
                }
        
        self.menu_clusters = cluster_groups
        self.cluster_features = cluster_characteristics
        
        # 클러스터 정보 출력
        for cluster_id, info in cluster_characteristics.items():
            logging.debug(f"클러스터 {cluster_id}: {info['menu_count']}개 메뉴, "
                  f"평균매출 {info['avg_sales']:.1f}, 타입: {info['dominant_type']}")
        
        return cluster_groups
    
    def _calculate_seasonality(self, group):
        """계절성 강도 계산"""
        if 'month' not in group.columns:
            return 0
        monthly_avg = group.groupby('month')['sales'].mean()
        if len(monthly_avg) < 2:
            return 0
        return monthly_avg.std() / (monthly_avg.mean() + 1e-8)
    
    def _calculate_trend(self, sales):
        """트렌드 강도 계산"""
        if len(sales) < 2:
            return 0
        x = np.arange(len(sales))
        return abs(np.polyfit(x, sales, 1)[0])
    
    def _get_dominant_menu_type(self, cluster_menus):
        """클러스터의 주요 메뉴 타입 결정"""
        type_counts = {'main': 0, 'drink': 0, 'premium': 0, 'group': 0, 'brunch': 0, 'other': 0}
        
        for store, menu in cluster_menus:
            menu_lower = str(menu).lower()
            if any(x in menu_lower for x in ['불고기', '갈비', '찌개', '국밥', '정식']):
                type_counts['main'] += 1
            elif any(x in menu_lower for x in ['콜라', '맥주', '소주', '커피', '음료']):
                type_counts['drink'] += 1
            elif any(x in menu_lower for x in ['한우', 'aus', '프리미엄']):
                type_counts['premium'] += 1
            elif '단체' in menu_lower:
                type_counts['group'] += 1
            elif '브런치' in menu_lower:
                type_counts['brunch'] += 1
            else:
                type_counts['other'] += 1
        
        return max(type_counts, key=type_counts.get)
    
    def get_menu_cluster(self, store, menu):
        """특정 메뉴의 클러스터 ID 반환"""
        for cluster_id, menus in self.menu_clusters.items():
            if (store, menu) in menus:
                return cluster_id
        return -1  # 클러스터에 없음
    
    def create_cluster_features(self, df, mode='train'):
        """클러스터 정보를 활용한 피처 생성"""
        
        sequences = []
        targets = []
        metadata = []
        
        for (store, menu), group in df.groupby(['store', 'menu']):
            group = group.sort_values('date').reset_index(drop=True)
            
            min_length = 28 + (7 if mode == 'train' else 0)
            if len(group) < min_length:
                continue
            
            # 클러스터 정보 가져오기
            cluster_id = self.get_menu_cluster(store, menu)
            cluster_info = self.cluster_features.get(cluster_id, {})
            
            if mode == 'predict':
                seq_data = group.tail(28)
                features = self._create_cluster_based_features(seq_data, store, menu, cluster_id, cluster_info)
                sequences.append(features)
                metadata.append({'store': store, 
                                 'menu': menu, 
                                 'cluster': cluster_id,
                                 'is_pre_launch_period': seq_data['is_pre_launch'].iloc[-1],
                                 'pre_launch_ratio': seq_data['is_pre_launch'].mean()})
                
                
            else:
                for i in range(len(group) - min_length + 1):
                    seq_data = group.iloc[i:i+28]
                    target_data = group.iloc[i+28:i+35]
                    
                    features = self._create_cluster_based_features(seq_data, store, menu, cluster_id, cluster_info)
                    target = target_data['sales'].values
                    
                    sequences.append(features)
                    targets.append(target)
                    metadata.append({'store': store, 
                                 'menu': menu, 
                                 'cluster': cluster_id,
                                 'is_pre_launch_period': seq_data['is_pre_launch'].mean(),
                                 'pre_launch_ratio': seq_data['is_pre_launch'].mean()})
        
        X = np.array(sequences) if sequences else np.empty((0, 60))
        y = np.array(targets) if targets else np.empty((0, 7))
        
        return X, y, metadata
    
    def _create_cluster_based_features(self, seq_data, store, menu, cluster_id, cluster_info):
        """클러스터 정보를 활용한 피처 생성"""
        
        sales = seq_data['sales'].values
        pre_launch_mask = seq_data['is_pre_launch'] == 1
        post_launch_mask = seq_data['is_pre_launch'] == 0
        
       # 1. 출시 상태 메타 피처
        launch_meta_features = [
            seq_data['is_pre_launch'].iloc[-1],      # 현재 출시 전 여부
            seq_data['is_pre_launch'].mean(),        # 시퀀스 내 출시 전 비율
            post_launch_mask.sum(),                  # 출시 후 일수
            pre_launch_mask.sum(),                   # 출시 전 일수
        ]
        # 2. 출시 후 데이터 통계 (핵심 - 실제 운영 패턴)
        if post_launch_mask.sum() > 0:
            post_sales = sales[post_launch_mask]
            post_launch_features = [
                np.mean(post_sales),                 # 출시 후 평균 매출
                np.std(post_sales),                  # 출시 후 변동성
                np.median(post_sales),               # 출시 후 중앙값
                np.max(post_sales),                  # 출시 후 최대값
                (post_sales == 0).mean(),            # 출시 후 실제 zero ratio
                np.mean(post_sales[-7:]) if len(post_sales) >= 7 else np.mean(post_sales),  # 최근 7일 평균
                post_sales[-1] if len(post_sales) > 0 else 0,  # 마지막 출시 후 매출
            ]
        else:
            post_launch_features = [0, 0, 0, 0, 1, 0, 0]  # 모두 출시 전

        # 3. 출시 전 데이터 통계 (참고용)
        if pre_launch_mask.sum() > 0:
            pre_sales = sales[pre_launch_mask]
            pre_launch_features = [
                np.mean(pre_sales),                  # 출시 전 평균 (보통 0)
                (pre_sales > 0).any(),               # 출시 전에도 매출이 있었는지
            ]
        else:
            pre_launch_features = [0, 0]  # 출시 전 없음
        
        # 4. 전체 시퀀스 패턴 피처 (출시 전/후 구분 없이)
        sequence_pattern_features = [
            len(sales),                              # 시퀀스 길이
            np.min(sales),                           # 전체 최소값 (보통 0)
            self._calculate_trend(sales),            # 전체 트렌드
            np.mean(sales[-3:]),                     # 최근 3일 평균
            (sales[-7:] > 0).sum() if len(sales) >= 7 else (sales > 0).sum(),  # 최근 7일 중 매출 있는 일수
        ]
        # 5. 출시 전/후 비교 피처 (패턴 변화 감지)
        if post_launch_mask.sum() > 0 and pre_launch_mask.sum() > 0:
            post_avg = np.mean(sales[post_launch_mask])
            pre_avg = np.mean(sales[pre_launch_mask])
            comparison_features = [
                post_avg - pre_avg,                  # 출시 후 매출 변화
                post_launch_mask.sum() / len(sales), # 출시 후 비율
            ]
        else:
            comparison_features = [0, post_launch_mask.sum() / len(sales)]
        
        # 6. 클러스터 기반 피처 (출시 후 데이터 기준으로 클러스터링)
        cluster_id = self.get_menu_cluster(store, menu)
        cluster_info = self.cluster_features.get(cluster_id, {})
        
        # 출시 후 평균과 클러스터 평균 비교
        post_avg = post_launch_features[0] if post_launch_features[0] > 0 else 0.01
        cluster_features = [
            cluster_id if cluster_id != -1 else 0,
            cluster_info.get('avg_sales', 0) / (post_avg + 1e-8),  # 클러스터 대비 성과
            cluster_info.get('zero_ratio', 0),
            cluster_info.get('volatility', 0),
        ]
        
        # 7. 시간 피처
        time_features = [
            seq_data['month'].iloc[-1],
            seq_data['day_of_week'].iloc[-1],
            seq_data['is_weekend'].sum(),
            np.sin(2 * np.pi * seq_data['month'].iloc[-1] / 12),
            np.cos(2 * np.pi * seq_data['month'].iloc[-1] / 12),
        ]
        
        # 8. 매장 피처
        store_features = self._get_store_features(store)

        
        
        # 모든 피처 결합 (중복 제거된 깔끔한 구조)
        all_features = (launch_meta_features +      # 4개
                       post_launch_features +       # 7개  
                       pre_launch_features +        # 2개
                       sequence_pattern_features +  # 5개
                       comparison_features +        # 2개
                       cluster_features +           # 4개
                       time_features +              # 5개
                       store_features)              # 4개
        
        # NaN 처리 및 크기 조정
        all_features = [0.0 if pd.isna(x) or np.isinf(x) else float(x) for x in all_features]
        
        # 총 33개 피처 - 더 효율적이고 명확함
        while len(all_features) < 35:
            all_features.append(0.0)
        all_features = all_features[:35]
        
        return [0.0 if pd.isna(x) or np.isinf(x) else float(x) for x in all_features]
    
    
    def _get_store_features(self, store):
        """업장 원-핫 인코딩"""
        stores = ['담하', '미라시아', '포레스트릿', '카페테리아', '화담숲주막', 
                 '화담숲카페', '느티나무 셀프BBQ', '연회장', '라그로타']
        return [int(store == s) for s in stores]
    
    def fit(self, train_df):
        """클러스터 기반 모델 학습"""
        logging.debug("클러스터 기반 모델 학습 시작...")
        
        # 1. 메뉴 출시일 감지
        self._detect_menu_launches(train_df)
        
        # 2. 출시 전 플래그 추가
        enhanced_df = self._add_pre_launch_flag(train_df)
        
        # 3. 수정된 zero ratio 계산
        self._calculate_corrected_zero_ratio(enhanced_df)
        
        # 4. 기존 클러스터 분석 (출시 후 데이터만)
        post_launch_df = enhanced_df[enhanced_df['is_pre_launch'] == 0]
        self.analyze_and_cluster_menus(post_launch_df)
        
        # 2. 클러스터 기반 피처 생성
        X, y, metadata = self.create_cluster_features(enhanced_df, mode='train')
        
        if len(X) == 0:
            logging.debug("학습 데이터 부족")
            return
        
        logging.debug(f"클러스터 피처: {X.shape}, 타겟: {y.shape}")
        
        # 3. 전체 모델 학습
        from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
        from sklearn.multioutput import MultiOutputRegressor
        from xgboost import XGBRegressor
        from lightgbm import LGBMRegressor
        # 6. Zero-inflation 분류기 (출시 후 시퀀스만)
        post_launch_sequences = []
        post_launch_targets = []
        
        for i, meta in enumerate(metadata):
            if meta['pre_launch_ratio'] < 0.3:  # 시퀀스의 30% 미만이 출시 전
                post_launch_sequences.append(i)
                post_launch_targets.append((y[i] > 0).any())
        
        # 매출 여부 분류기
        self.zero_classifier = RandomForestClassifier(
            n_estimators=200, max_depth=12, random_state=42, class_weight='balanced'
        )
        if len(post_launch_sequences) > 0:
            from sklearn.ensemble import RandomForestClassifier
            self.zero_classifier = RandomForestClassifier(
                n_estimators=200, max_depth=12, random_state=42, class_weight='balanced'
            )
            
            X_post = X[post_launch_sequences]
            y_post = np.array(post_launch_targets)

            self.zero_classifier.fit(X_post, y_post)
            print(f"Zero 분류기: {len(post_launch_sequences)}개 출시 후 시퀀스로 학습")
            print(f"출시 후 시퀀스 zero ratio: {(~y_post).mean():.3f}")

        # 7. 회귀 모델 (전체 데이터, 출시 전에 낮은 가중치)
        has_sales = (y > 0).any(axis=1)
        sales_mask = has_sales
    
        # 4. 클러스터별 전용 모델 학습
        sales_mask = has_sales

        if sales_mask.sum() > 20:
            
            from sklearn.multioutput import MultiOutputRegressor
            from lightgbm import LGBMRegressor
            
            self.global_model = MultiOutputRegressor(
                LGBMRegressor(n_estimators=600, max_depth=8, learning_rate=0.05,
                            random_state=42, verbosity=-1)
            )
            self.global_model.fit(X[sales_mask], y[sales_mask])
            print(f"회귀 모델: {sales_mask.sum()}개 시퀀스, 출시 전 가중치 적용")
        
            
            # 클러스터별 모델 학습 (가장 유용한 부분)
            logging.debug("클러스터별 전용 모델 학습 중...")
            self.cluster_models = {}
            
            # 학습할 클러스터들 필터링
            valid_clusters = []
            for cluster_id in self.menu_clusters.keys():
                cluster_mask = np.array([meta['cluster'] == cluster_id for meta in metadata])
                cluster_sales_mask = sales_mask & cluster_mask
                if cluster_sales_mask.sum() > 10:
                    valid_clusters.append((cluster_id, cluster_sales_mask))

            # tqdm으로 클러스터별 모델 학습 진행률 표시
            for cluster_id, cluster_sales_mask in tqdm(valid_clusters, desc="클러스터 모델"):
                from xgboost import XGBRegressor
                
                cluster_model = MultiOutputRegressor(
                    XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                            random_state=42, verbosity=0)
                )
                
                cluster_model.fit(X[cluster_sales_mask], y[cluster_sales_mask])
                self.cluster_models[cluster_id] = cluster_model
                
                # 클러스터 정보 표시
                cluster_info = self.cluster_features[cluster_id]
                tqdm.write(f"클러스터 {cluster_id} 완료: {cluster_info['dominant_type']}, "
                        f"{cluster_sales_mask.sum()}개 샘플")
        
        logging.debug(f"전체 모델 + {len(self.cluster_models)}개 클러스터 모델 학습 완료")
    
    def predict(self, test_df):
        """클러스터 기반 예측"""
        if not hasattr(self, 'global_model'):
            logging.debug("모델이 학습되지 않음")
            return np.zeros((len(test_df), 7)), []
        
        enhanced_test_df = self._add_pre_launch_flag(test_df)

        X, _, metadata = self.create_cluster_features(enhanced_test_df, mode='predict')
        
        if len(X) == 0:
            return np.zeros((0, 7)), []
        
        logging.debug("매출 여부 예측 중...")
        with tqdm(total=100, desc="분류 예측") as pbar:
            has_sales_prob = self.zero_classifier.predict_proba(X)[:, 1]
            pbar.update(100)
        
        # 예측 수행
        predictions = np.zeros((len(X), 7))
        
        for i in tqdm(range(len(X)), desc="예측 진행"):
            cluster_id = metadata[i]['cluster']
            
            # 클러스터 전용 모델이 있으면 사용, 없으면 전체 모델 사용
            if cluster_id in self.cluster_models:
                cluster_pred = self.cluster_models[cluster_id].predict(X[i:i+1])
                global_pred = self.global_model.predict(X[i:i+1])
                
                # 클러스터 모델과 전체 모델의 가중 평균
                cluster_weight = 0.7
                predictions[i] = cluster_weight * cluster_pred[0] + (1-cluster_weight) * global_pred[0]
            else:
                # 클러스터 모델이 없으면 전체 모델만 사용
                predictions[i] = self.global_model.predict(X[i:i+1])[0]
        
        # Zero-inflation 적용
        threshold = self.corrected_zero_ratio
        zero_mask = has_sales_prob < threshold
        predictions[zero_mask] = 0
        
        # 후처리
        predictions = np.maximum(predictions, 1)
        # predictions = np.round(predictions, 2)
        
        return predictions, metadata

# 실행 함수
def run_cluster_based_pipeline(df:pd.DataFrame):
    """클러스터 기반 파이프라인 실행"""
    
    train_df = df
    # 데이터 로드
    train_df['date'] = pd.to_datetime(train_df['영업일자'])
    train_df[['store', 'menu']] = train_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
    train_df['sales'] = train_df['매출수량']
    train_df['month'] = train_df['date'].dt.month
    train_df['day_of_week'] = train_df['date'].dt.dayofweek
    train_df['is_weekend'] = train_df['day_of_week'].isin([5, 6])
    
    # 클러스터 기반 모델 학습
    cluster_model = ClusterBasedForecastingModel()
    cluster_model.fit(train_df)
    
    # 제출 파일 생성
    submission = pd.read_csv('sample_submission.csv')
    
    import glob
    test_files = sorted(glob.glob('TEST_*.csv'))
    
    for test_idx, test_file in enumerate(test_files):
        logging.debug(f"클러스터 기반 처리: {test_file}")
        
        test_df = pd.read_csv(test_file)
        test_df['date'] = pd.to_datetime(test_df['영업일자'])
        test_df[['store', 'menu']] = test_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
        test_df['sales'] = test_df['매출수량']
        test_df['month'] = test_df['date'].dt.month
        test_df['day_of_week'] = test_df['date'].dt.dayofweek
        test_df['is_weekend'] = test_df['day_of_week'].isin([5, 6])
        
        # 예측
        predictions, metadata = cluster_model.predict(test_df)
        
        # 제출 파일에 매핑
        test_case = f"TEST_{test_idx:02d}"
        test_rows = submission[submission['영업일자'].str.contains(test_case, na=False)].index.tolist()
        
        if len(test_rows) == 7 and len(predictions) > 0:
            numeric_cols = submission.select_dtypes(include=[np.number]).columns
            
            for day_idx, row_idx in enumerate(test_rows):
                for col_idx, col in enumerate(numeric_cols):
                    if col_idx < len(predictions):
                        pred_value = predictions[col_idx, day_idx]
                        submission.loc[row_idx, col] = max(0.0, pred_value)
    
    # submission.to_csv('cluster_based_submission.csv', index=False)
    logging.debug("cluster_based_submission.csv 생성 완료!")
    
    return submission

# 실행
#cluster_submission = run_cluster_based_pipeline

In [38]:

# 데이터 로드 (예시)
df = pd.read_csv('train/train.csv')
train = df
cluster_submission = run_cluster_based_pipeline(train)
cluster_submission.to_csv('./Submission/submission_10.csv', index=False, encoding='utf-8-sig')


2025-08-22 21:32:24,006 DEBUG 클러스터 기반 모델 학습 시작...


메뉴 출시일 분석 중...
느티나무 셀프BBQ_신라면: 103일 출시 전 → 2023-04-14 00:00:00
느티나무 셀프BBQ_쌈장: 103일 출시 전 → 2023-04-14 00:00:00
느티나무 셀프BBQ_육개장 사발면: 103일 출시 전 → 2023-04-14 00:00:00
느티나무 셀프BBQ_잔디그늘집 대여료 (12인석): 103일 출시 전 → 2023-04-14 00:00:00
느티나무 셀프BBQ_잔디그늘집 의자 추가: 103일 출시 전 → 2023-04-14 00:00:00
느티나무 셀프BBQ_햇반: 103일 출시 전 → 2023-04-14 00:00:00
느티나무 셀프BBQ_허브솔트: 103일 출시 전 → 2023-04-14 00:00:00
담하_(단체) 생목살 김치전골 2.0: 260일 출시 전 → 2023-09-18 00:00:00
담하_(단체) 은이버섯 갈비탕: 162일 출시 전 → 2023-06-12 00:00:00
담하_(정식) 된장찌개: 153일 출시 전 → 2023-06-03 00:00:00
담하_(정식) 물냉면 : 153일 출시 전 → 2023-06-03 00:00:00
담하_(정식) 비빔냉면: 153일 출시 전 → 2023-06-03 00:00:00
담하_(후식) 물냉면: 152일 출시 전 → 2023-06-02 00:00:00
담하_(후식) 비빔냉면: 152일 출시 전 → 2023-06-02 00:00:00
담하_갱시기: 341일 출시 전 → 2023-12-08 00:00:00
담하_꼬막 비빔밥: 250일 출시 전 → 2023-09-08 00:00:00
담하_담하 한우 불고기 정식: 152일 출시 전 → 2023-06-02 00:00:00
담하_더덕 한우 지짐: 251일 출시 전 → 2023-09-09 00:00:00
담하_명인안동소주: 181일 출시 전 → 2023-07-01 00:00:00
담하_명태회 비빔냉면: 152일 출시 전 → 2023-06-02 00:00:00
담하_문막 복분자 칵테일: 254일 출시 전 → 

2025-08-22 21:32:24,631 DEBUG 메뉴 클러스터링 시작...


출시 전 플래그 적용: 10,194개 (9.9%)
수정된 통계:
  출시 후 데이터: 92,482개
  0 매출: 43,840개
  실제 zero ratio: 0.474 (기존 threshold 0.3 vs 수정 0.474)


메뉴 특성 추출: 100%|██████████| 193/193 [00:00<00:00, 1474.47it/s]
2025-08-22 21:32:24,784 DEBUG 클러스터 수: 6
클러스터 특성 분석: 100%|██████████| 6/6 [00:01<00:00,  5.98it/s]
2025-08-22 21:32:25,812 DEBUG 클러스터 0: 38개 메뉴, 평균매출 21.9, 타입: other
2025-08-22 21:32:25,812 DEBUG 클러스터 1: 21개 메뉴, 평균매출 9.1, 타입: main
2025-08-22 21:32:25,812 DEBUG 클러스터 2: 12개 메뉴, 평균매출 20.9, 타입: group
2025-08-22 21:32:25,813 DEBUG 클러스터 3: 7개 메뉴, 평균매출 21.1, 타입: brunch
2025-08-22 21:32:25,813 DEBUG 클러스터 4: 111개 메뉴, 평균매출 2.8, 타입: other
2025-08-22 21:32:25,814 DEBUG 클러스터 5: 4개 메뉴, 평균매출 106.5, 타입: other
2025-08-22 21:32:57,920 DEBUG 클러스터 피처: (96114, 35), 타겟: (96114, 7)


...

예측 진행: 100%|██████████| 193/193 [00:00<00:00, 309.85it/s]
2025-08-22 21:33:46,737 DEBUG 클러스터 기반 처리: TEST_08.csv
2025-08-22 21:33:46,857 DEBUG 매출 여부 예측 중...


출시 전 플래그 적용: 0개 (0.0%)


예측 진행: 100%|██████████| 193/193 [00:00<00:00, 309.43it/s]
2025-08-22 21:33:47,738 DEBUG 클러스터 기반 처리: TEST_09.csv
2025-08-22 21:33:47,857 DEBUG 매출 여부 예측 중...


출시 전 플래그 적용: 0개 (0.0%)


예측 진행: 100%|██████████| 193/193 [00:00<00:00, 309.48it/s]
2025-08-22 21:33:48,733 DEBUG cluster_based_submission.csv 생성 완료!


In [17]:
import pandas as pd
import numpy as np

# 파일 로드
print("파일 로드 중...")
df = pd.read_csv('./Submission/submission_4.csv')

print(f"원본 데이터: {len(df)}행")

# 화담숲 관련 컬럼 찾기
forest_columns = [col for col in df.columns if '화담숲주막' in col or '화담숲카페' in col]
print(f"화담숲 관련 컬럼: {len(forest_columns)}개")
print(f"컬럼들: {forest_columns}")

# 수정 전 상태 확인
print("\n=== 수정 전 상태 ===")
for test_name in ['TEST_05', 'TEST_06', 'TEST_07']:
    test_mask = df['영업일자'].str.contains(test_name, na=False)
    test_data = df[test_mask]
    
    if len(test_data) > 0:
        total_before = test_data[forest_columns].sum().sum()
        print(f"{test_name}: 화담숲 전체 합계 {total_before:.2f}")

# 직접 수정: TEST_05, TEST_06, TEST_07에서 화담숲 매장들을 0으로 설정
print("\n=== 수정 실행 ===")
modified_count = 0

for test_name in ['TEST_05', 'TEST_06', 'TEST_07']:
    print(f"{test_name} 수정 중...")
    
    # 해당 TEST의 모든 행 찾기
    test_mask = df['영업일자'].str.contains(test_name, na=False)
    test_indices = df[test_mask].index
    
    print(f"  {test_name} 행 수: {len(test_indices)}")
    
    # 화담숲 컬럼들을 0으로 설정
    for idx in test_indices:
        for col in forest_columns:
            original_value = df.loc[idx, col]
            if pd.notna(original_value) and original_value > 0:
                df.loc[idx, col] = 0.0
                modified_count += 1
    
    print(f"  {test_name} 완료")

print(f"\n총 {modified_count}개 값이 0으로 수정됨")

# 수정 후 상태 확인
print("\n=== 수정 후 상태 ===")
for test_name in ['TEST_05', 'TEST_06', 'TEST_07']:
    test_mask = df['영업일자'].str.contains(test_name, na=False)
    test_data = df[test_mask]
    
    if len(test_data) > 0:
        total_after = test_data[forest_columns].sum().sum()
        print(f"{test_name}: 화담숲 전체 합계 {total_after:.2f}")

# 수정된 파일 저장
output_file = 'submission_4_FIXED.csv'
df.to_csv(output_file, index=False)
print(f"\n수정된 파일 저장: {output_file}")

# 검증
print("\n=== 검증 ===")
verification_df = pd.read_csv(output_file)

for test_name in ['TEST_05', 'TEST_06', 'TEST_07']:
    test_mask = verification_df['영업일자'].str.contains(test_name, na=False)
    test_data = verification_df[test_mask]
    
    if len(test_data) > 0:
        total_check = test_data[forest_columns].sum().sum()
        success = total_check == 0.0
        print(f"{test_name}: 합계 {total_check:.2f} - {'성공' if success else '실패'}")

print("\n완료!")

파일 로드 중...
원본 데이터: 70행
화담숲 관련 컬럼: 13개
컬럼들: ['화담숲주막_느린마을 막걸리', '화담숲주막_단호박 식혜 ', '화담숲주막_병천순대', '화담숲주막_스프라이트', '화담숲주막_참살이 막걸리', '화담숲주막_찹쌀식혜', '화담숲주막_콜라', '화담숲주막_해물파전', '화담숲카페_메밀미숫가루', '화담숲카페_아메리카노 HOT', '화담숲카페_아메리카노 ICE', '화담숲카페_카페라떼 ICE', '화담숲카페_현미뻥스크림']

=== 수정 전 상태 ===
TEST_05: 화담숲 전체 합계 91.00
TEST_06: 화담숲 전체 합계 91.00
TEST_07: 화담숲 전체 합계 91.00

=== 수정 실행 ===
TEST_05 수정 중...
  TEST_05 행 수: 7
  TEST_05 완료
TEST_06 수정 중...
  TEST_06 행 수: 7
  TEST_06 완료
TEST_07 수정 중...
  TEST_07 행 수: 7
  TEST_07 완료

총 273개 값이 0으로 수정됨

=== 수정 후 상태 ===
TEST_05: 화담숲 전체 합계 0.00
TEST_06: 화담숲 전체 합계 0.00
TEST_07: 화담숲 전체 합계 0.00

수정된 파일 저장: submission_4_FIXED.csv

=== 검증 ===
TEST_05: 합계 0.00 - 성공
TEST_06: 합계 0.00 - 성공
TEST_07: 합계 0.00 - 성공

완료!
